# PWMV Variants Comparison: standard / gated / dual-probe

Follow-up ao validation run n=30 seed=777 que falhou (Δ=0 em 5 checkpoints consecutivos). Hipótese: PWMV não diferencia quando 4 traces convergem pro mesmo answer. Testa 3 variants + baselines no MESMO sample.

## Variants

1. **greedy** — baseline
2. **naive_vote** — self-consistency N=4
3. **pwmv** — LiLaVe-style (probe weighted)
4. **gated_pwmv** — CISC-variant: PWMV só se naive_vote não tem clear winner (frac_max < 0.75). Senão retorna naive.
5. **dual_probe** — GENUINELY NOVEL: train fresh L23 LogReg, combine L11 × L23 scores (multiplicative agreement). Goodhart-like: both probes must agree.

## Diagnostic logging

Per prompt loga:
- `unique_answers_count` (1-4)
- `divergence_flag` (convergent vs divergent)
- Per-method answer + correctness

**Breakdown final**: mostra acc por subset (divergent vs convergent). Hipótese: divergent subset mostra +6pp PWMV, convergent subset = Δ=0.

## Prior art acknowledgment

- **PWMV**: LiLaVe arxiv:2504.16760 (canonical reference, we cite)
- **Gated PWMV**: CISC arxiv:2502.06233 (confidence tie-breaker, our version uses probe not token-prob — narrow delta)
- **Dual-probe**: first attempt at adversarial agreement, no direct prior art

Budget: ~3.5h on RTX 6000.

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path
def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception: ok = False
if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer')
    pip('uninstall', '-y', 'transformers', 'causal-conv1d')
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    pip('install','--no-cache-dir','flash-linear-attention')
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'): del sys.modules[m]

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, re, time, random, pickle
import numpy as np
import torch
from tqdm.auto import tqdm
from collections import defaultdict, Counter
from safetensors import safe_open
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from huggingface_hub import snapshot_download, HfApi, create_repo
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/pwmv_variants'); OUT.mkdir(exist_ok=True)
MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'
HF_PROBE = 'caiovicentino1/qwen36-deepconf-probe'
HF_STAGE_B = 'caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b'
HF_TARGET = 'caiovicentino1/qwen36-pwmv-variants'  # new repo for this experiment

N_TRACES = 4
TEMP = 0.7
MAX_NEW = 3500
N_PROMPTS = 30
SEED = 777  # same seed as failed validation - see if variants rescue
GATE_THRESHOLD = 0.75  # gated variant: if max_answer_frac >= 0.75, use naive (clear winner)
FORCE_SUFFIX = '\n</think>\n\nFinal answer: \\boxed{'

print('setup done — 30 prompts seed 777, same as failed validation')

## Cell 2 — Load Qwen3.6 + install L11 + L23 hooks

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print(f'Model: {torch.cuda.memory_allocated()/1e9:.1f} GB')

def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

# Two hooks: L11 (existing probe) + L23 (will train fresh probe for dual-probe variant)
captured = {11: [], 23: []}
def make_hook(L):
    def h(mod, inp, out):
        x = out[0] if isinstance(out, tuple) else out
        captured[L].append(x[:, -1, :].detach().float().cpu())
    return h

handles = []
for L in [11, 23]:
    layer = get_layer_module(model, L)
    handles.append(layer.register_forward_hook(make_hook(L)))
print(f'OK {len(handles)} hooks installed (L11, L23)')

## Cell 3 — Load L11 probe + train fresh L23 probe from Stage B

In [ ]:
# L11 probe from HF
probe_dir = snapshot_download(HF_PROBE, repo_type='model', cache_dir='/content/cache',
                               allow_patterns=['probe_l11.pkl'])
with open(Path(probe_dir)/'probe_l11.pkl', 'rb') as f:
    probe_l11 = pickle.load(f)
print('L11 probe loaded (AUROC 0.78)')

# L23 probe: train fresh from Stage B rollouts
corpus = snapshot_download(HF_STAGE_B, repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

X_l23, y = [], []
for shard in tqdm(shards, desc='load Stage B L23'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        rs = json.loads(meta['rollouts'])
        keys = list(f.keys())
        for r in rs:
            if r.get('pred') is None: continue
            cands = [k for k in keys if 'l23' in k.lower()]
            if not cands: continue
            act = f.get_tensor(cands[0])
            if act.dim() == 2: emb = act.float().mean(dim=0).numpy()
            elif act.dim() == 1: emb = act.float().numpy()
            elif act.dim() == 3: emb = act.float().reshape(-1, act.shape[-1]).mean(dim=0).numpy()
            else: continue
            X_l23.append(emb); y.append(int(r.get('correct', False)))
            break

X_l23 = np.stack(X_l23); y = np.array(y)
print(f'L23 probe data: {X_l23.shape}, correct rate {y.mean():.3f}')

X_tr, X_va, y_tr, y_va = train_test_split(X_l23, y, test_size=0.3, random_state=42, stratify=y)
probe_l23 = LogisticRegression(max_iter=2000, C=0.5, class_weight='balanced')
probe_l23.fit(X_tr, y_tr)
l23_auc = roc_auc_score(y_va, probe_l23.predict_proba(X_va)[:, 1])
print(f'L23 probe val AUROC: {l23_auc:.3f}')
with open(OUT / 'probe_l23.pkl', 'wb') as f: pickle.dump(probe_l23, f)

## Cell 4 — Load eval prompts (SAME seed 777 as failed validation)

In [ ]:
questions = []
seen = set()
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q = meta['question']
        if q in seen: continue
        seen.add(q)
        opts = json.loads(meta['options'])
        if len(opts) != 10: continue
        questions.append(dict(question=q, options=opts, gold_letter=meta['gold_letter']))

# Same exclusion logic as validation: skip original seed 123 first 50
random.seed(123)
questions_123 = questions.copy()
random.shuffle(questions_123)
used = set(q['question'] for q in questions_123[:50])

random.seed(SEED)
fresh = [q for q in questions if q['question'] not in used]
random.shuffle(fresh)
eval_set = fresh[:N_PROMPTS]
print(f'{len(eval_set)} eval prompts (seed {SEED}, disjoint from original)')

## Cell 5 — Helpers + 5-variant generator

In [ ]:
def extract_answer(text):
    post = text.split('</think>')[-1] if '</think>' in text else text
    for pattern in [r'\\boxed\{([A-J])\}',
                    r'[Aa]nswer\s*(?:is\s*)?[:=]?\s*\*?\*?\(?([A-J])\)?\*?\*?',
                    r'^\s*\(?([A-J])\)?\s*\.?\s*$']:
        m = re.search(pattern, post, re.MULTILINE)
        if m: return m.group(1).upper()
    m = re.findall(r'\\boxed\{([A-J])\}', text)
    if m: return m[-1]
    tail = post[-300:] if post else text[-300:]
    letters = re.findall(r'\b([A-J])\b', tail)
    return letters[-1] if letters else None

def fmt(q, opts):
    choices = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(opts))
    content = (f'Answer the following multiple-choice question. '
               f'Reason step by step, then put the letter in \\boxed{{}}.\n\n'
               f'Question: {q}\n\nOptions:\n{choices}')
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=True)

def force_close(full_ids, prompt_len):
    gen = tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=False)
    if '</think>' in gen:
        return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True)
    full = tok.decode(full_ids.tolist(), skip_special_tokens=False) + FORCE_SUFFIX
    ctx = tok(full, return_tensors='pt', add_special_tokens=False).input_ids.cuda()
    with torch.no_grad():
        out = model.generate(ctx, max_new_tokens=15, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    suf = tok.decode(out[0, ctx.shape[1]:].tolist(), skip_special_tokens=True)
    return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True) + FORCE_SUFFIX + suf

def gen_greedy(ids):
    captured[11] = []; captured[23] = []
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    return extract_answer(force_close(out[0], ids.shape[1]))

def gen_all_variants(ids, n_traces=N_TRACES, temp=TEMP):
    '''Returns dict with 4 methods + diagnostic info.'''
    captured[11] = []; captured[23] = []
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=True, temperature=temp,
                             num_return_sequences=n_traces, top_p=0.95,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    # Mean-pool per-trace
    def mean_per_trace(chunks):
        if len(chunks) < 2: return None
        # chunks[0] is prompt forward (shape [n_traces, d]), chunks[1:] are gen tokens
        stack = torch.stack(chunks[1:], dim=0)  # [gen_len, n_traces, d]
        return stack.mean(dim=0).numpy()  # [n_traces, d]
    embs_11 = mean_per_trace(captured[11])
    embs_23 = mean_per_trace(captured[23])
    # Probe scores
    scores_11 = probe_l11.predict_proba(embs_11)[:, 1].tolist() if embs_11 is not None else [1.0]*n_traces
    scores_23 = probe_l23.predict_proba(embs_23)[:, 1].tolist() if embs_23 is not None else [1.0]*n_traces
    # Extract answers
    answers = [extract_answer(force_close(out[i], ids.shape[1])) for i in range(n_traces)]

    # Diagnostic
    valid = [a for a in answers if a]
    counts = Counter(valid)
    unique = len(counts)
    total_valid = len(valid)
    max_frac = counts.most_common(1)[0][1] / total_valid if total_valid else 0
    divergent = unique >= 2

    # Method 1: naive vote
    naive = counts.most_common(1)[0][0] if counts else None

    # Method 2: pwmv (LiLaVe)
    w_pwmv = defaultdict(float)
    for a, s in zip(answers, scores_11):
        if a: w_pwmv[a] += s
    pwmv = max(w_pwmv, key=w_pwmv.get) if w_pwmv else None

    # Method 3: gated_pwmv (CISC-style with probe)
    if max_frac >= GATE_THRESHOLD:
        gated = naive  # clear winner, skip probe
    else:
        gated = pwmv  # ambiguous, use probe

    # Method 4: dual_probe (multiplicative agreement between L11 and L23)
    w_dual = defaultdict(float)
    for a, s11, s23 in zip(answers, scores_11, scores_23):
        if a: w_dual[a] += s11 * s23  # Goodhart-guarded: both must agree
    dual = max(w_dual, key=w_dual.get) if w_dual else None

    return {
        'naive': naive, 'pwmv': pwmv, 'gated': gated, 'dual': dual,
        'answers': answers,
        'scores_l11': scores_11, 'scores_l23': scores_23,
        'unique_answers': unique, 'max_frac': max_frac, 'divergent': divergent,
    }

print('helpers ready')

## Cell 6 — Eval 5 methods × 30 prompts

In [ ]:
results = {'greedy': [], 'naive': [], 'pwmv': [], 'gated': [], 'dual': []}
diag = []
t0 = time.time()

for qi, q in enumerate(tqdm(eval_set, desc='eval variants')):
    p = fmt(q['question'], q['options'])
    ids = tok(p, return_tensors='pt').input_ids.cuda()
    if ids.shape[1] > 1536: continue
    gold = q['gold_letter']

    # Greedy
    try:
        g = gen_greedy(ids); results['greedy'].append((g, g == gold))
    except Exception as e:
        print(f'greedy err: {e}'); results['greedy'].append((None, False))

    # 4-trace + 4 voting variants
    try:
        v = gen_all_variants(ids)
        for m in ['naive', 'pwmv', 'gated', 'dual']:
            ans = v[m]
            results[m].append((ans, ans == gold))
        diag.append({
            'qi': qi, 'gold': gold, 'greedy': g,
            'naive': v['naive'], 'pwmv': v['pwmv'], 'gated': v['gated'], 'dual': v['dual'],
            'unique_answers': v['unique_answers'], 'max_frac': v['max_frac'],
            'divergent': v['divergent'], 'answers': v['answers'],
            'scores_l11': v['scores_l11'], 'scores_l23': v['scores_l23'],
        })
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        for m in ['naive', 'pwmv', 'gated', 'dual']: results[m].append((None, False))

    if (qi+1) % 3 == 0:
        print(f'  [{qi+1}/{len(eval_set)}]', end=' ')
        for m in ['greedy','naive','pwmv','gated','dual']:
            acc = np.mean([r[1] for r in results[m]])
            print(f'{m}={acc:.3f}', end=' ')
        n_div = sum(1 for d in diag if d['divergent'])
        print(f' | div={n_div}/{len(diag)}')
        # Crash-safe save
        with open(OUT / 'partial.json', 'w') as f:
            partial = {k: [(r[0], bool(r[1])) for r in v] for k, v in results.items()}
            json.dump({'results': partial, 'diag': diag}, f, indent=2)

for h in handles: h.remove()

print(f'\n=== FINAL (n={len(eval_set)}, {(time.time()-t0)/60:.0f} min) ===')
for m in ['greedy','naive','pwmv','gated','dual']:
    acc = np.mean([r[1] for r in results[m]])
    print(f'  {m:8s}: {acc:.3f}')

# Breakdown by divergence
n_div = sum(1 for d in diag if d['divergent'])
n_conv = sum(1 for d in diag if not d['divergent'])
print(f'\n=== BREAKDOWN BY DIVERGENCE ===')
print(f'Divergent prompts: {n_div}/{len(diag)} | Convergent: {n_conv}')

for subset, flag in [('divergent', True), ('convergent', False)]:
    mask = [d['divergent'] == flag for d in diag]
    if not any(mask): continue
    print(f'\n  [{subset}] n={sum(mask)}')
    for m in ['greedy','naive','pwmv','gated','dual']:
        subset_results = [r[1] for r, d in zip(results[m][:len(diag)], diag) if d['divergent'] == flag]
        if not subset_results: continue
        acc = np.mean(subset_results)
        print(f'    {m:8s}: {acc:.3f}')

## Cell 7 — Upload results + diagnostic

In [ ]:
summary = {
    'model': MODEL_ID, 'n_prompts': len(eval_set), 'seed': SEED,
    'n_traces': N_TRACES, 'temperature': TEMP, 'max_new_tokens': MAX_NEW,
    'gate_threshold': GATE_THRESHOLD,
    'probe_l11_source': HF_PROBE,
    'probe_l23_val_auroc': float(l23_auc),
    'results': {m: float(np.mean([r[1] for r in results[m]])) for m in results},
    'deltas_vs_greedy_pp': {m: float((np.mean([r[1] for r in results[m]]) - np.mean([r[1] for r in results['greedy']]))*100)
                             for m in ['naive','pwmv','gated','dual']},
    'n_divergent': sum(1 for d in diag if d['divergent']),
    'n_convergent': sum(1 for d in diag if not d['divergent']),
    'prior_art': {
        'pwmv': 'LiLaVe arxiv:2504.16760',
        'gated_pwmv': 'CISC arxiv:2502.06233 (tie-break with confidence, ours uses probe)',
        'dual_probe': 'Novel — no direct prior art for Goodhart-guarded probe agreement',
    },
}
with open(OUT / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
with open(OUT / 'diag.json', 'w') as f:
    json.dump(diag, f, indent=2)

api = HfApi()
create_repo(HF_TARGET, repo_type='model', private=False, exist_ok=True)
for fn in ['summary.json', 'diag.json', 'probe_l23.pkl', 'partial.json']:
    fp = OUT / fn
    if fp.exists():
        api.upload_file(path_or_fileobj=str(fp), path_in_repo=fn,
                        repo_id=HF_TARGET, repo_type='model',
                        commit_message=f'PWMV variants exp: {fn}')
        print(f'OK {fn}')
print(f'\n✅ Results: https://huggingface.co/{HF_TARGET}')